In [2]:
def main(datasource, start_date, end_date):
    """
    factor function - optimized version

    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format

    Returns:
        pd.DataFrame: Factor data with columns ['date', 'instrument', 'factor']
    """
    import dai
    import calendar
    import numpy as np
    import pandas as pd
    from datetime import date, datetime

    # 预计算常数
    WEIGHTS = np.exp(-0.3 * np.arange(5))
    WEIGHTS = WEIGHTS / WEIGHTS.sum()
    
    def calculate_weighted_imbalance_vectorized(df):
        """向量化计算加权订单簿不平衡度"""
        bid_cols = [f'bid_volume{i}' for i in range(1, 6)]
        ask_cols = [f'ask_volume{i}' for i in range(1, 6)]
        
        for col in bid_cols + ask_cols:
            if col not in df.columns:
                df[col] = 0

        weighted_bid = np.sum(df[bid_cols].values * WEIGHTS, axis=1)
        weighted_ask = np.sum(df[ask_cols].values * WEIGHTS, axis=1)
        
        total = weighted_bid + weighted_ask + 1e-8
        imbalance = (weighted_bid - weighted_ask) / total
        
        return imbalance

    def calculate_depth_ratio_vectorized(df):
        """向量化计算深度比率"""
        # 使用前3档
        bid_cols = [f'bid_volume{i}' for i in range(1, 4)]
        ask_cols = [f'ask_volume{i}' for i in range(1, 4)]
        
        # 确保所有列都存在
        for col in bid_cols + ask_cols:
            if col not in df.columns:
                df[col] = 0
        
        bid_depth = df[bid_cols].sum(axis=1).values
        ask_depth = df[ask_cols].sum(axis=1).values
        
        total = bid_depth + ask_depth + 1e-8
        depth_ratio = (bid_depth - ask_depth) / total
        
        return depth_ratio

    def calculate_window_factor_optimized(window_df):
        """优化单个15分钟窗口的因子计算"""
        n = len(window_df)
        if n < 10:
            return np.nan
        
        raw_pressure = window_df['raw_pressure'].values
        mid_prices = window_df['mid_price'].values
        
        # 计算统计量
        raw_pressure_mean = np.mean(raw_pressure)
        raw_pressure_std = np.std(raw_pressure)
        
        # 计算波动率
        if n > 1:
            log_returns = np.log(mid_prices[1:] / mid_prices[:-1])
            volatility = np.std(log_returns)
        else:
            volatility = 0
        
        # 标准化原始压力因子
        if raw_pressure_std > 0:
            standardized_pressure = (raw_pressure[-1] - raw_pressure_mean) / raw_pressure_std
        else:
            standardized_pressure = 0
        
        # 根据波动率调整因子
        if n > 10:
            pct_changes = np.abs(np.diff(mid_prices) / mid_prices[:-1])
            volatility_threshold = np.percentile(pct_changes, 80)
        else:
            volatility_threshold = 0.01
        
        if volatility > volatility_threshold:
            adjusted_pressure = standardized_pressure * 0.7
        else:
            adjusted_pressure = standardized_pressure
        
        # 应用非线性变换
        return np.tanh(adjusted_pressure)

    def label_time_segment_optimized(df, date_col='date'):
        """优化时间段标签计算"""
        # 使用小时和分钟进行计算
        hours = df[date_col].dt.hour.values
        minutes = df[date_col].dt.minute.values
        
        total_minutes = hours * 60 + minutes
        
        # 预定义边界和标签
        bins = np.array([
            0, 9*60+30, 9*60+45, 10*60+0, 10*60+15, 10*60+30, 10*60+45,
            11*60+0, 11*60+15, 11*60+30, 13*60+0, 13*60+15, 13*60+30,
            13*60+45, 14*60+0, 14*60+15, 14*60+30, 14*60+45, 14*60+57, np.inf
        ])
        
        labels = [
            'OTHERS', '09:45:00', '10:00:00', '10:15:00', '10:30:00', 
            '10:45:00', '11:00:00', '11:15:00', '11:30:00', 'OTHERS1',
            '13:15:00', '13:30:00', '13:45:00', '14:00:00', '14:15:00',
            '14:30:00', '14:45:00', '15:00:00', 'OTHERS2'
        ]
        
        # 使用numpy的digitize进行快速分箱
        indices = np.digitize(total_minutes, bins) - 1
        result = pd.Series([labels[i] for i in indices], index=df.index)
        
        return result

    def calc_factor(df):
        """因子计算"""
        df.fillna(0, inplace=True)
        
        # 向量化日期转换
        df['trading_day'] = df['date'].dt.strftime('%Y%m%d').astype(int)
        df['time'] = df['date'].dt.strftime('%H%M%S').astype(int)
        
        # 剔除集合竞价数据
        mask = (df['time'] >= 93000) & (df['time'] <= 150000)
        df = df[mask].copy()
        
        # 分组计算成交量差 - 使用transform提高性能
        df['volume'] = df.groupby(['instrument_id', 'trading_day'])['volume'].transform(
            lambda x: x.diff()
        )
        
        # 计算中间价
        # df['mid_price'] = (df['ask_price1'] + df['bid_price1']) * 0.5
        def calculate_mid_price(bid, ask):
            if ask == 0:
                return bid
            elif bid == 0:
                return ask
            else:
                return (bid + ask) * 0.5
        df['mid_price'] = np.vectorize(calculate_mid_price)(df['bid_price1'], df['ask_price1'])

        # 向量化计算各种指标
        df['imbalance'] = calculate_weighted_imbalance_vectorized(df)
        df['relative_spread'] = (df['ask_price1'] - df['bid_price1']) / df['mid_price']
        df['depth_ratio'] = calculate_depth_ratio_vectorized(df)
        
        # 计算原始压力因子
        df['spread_sqrt'] = np.sqrt(df['relative_spread'].abs() + 1e-8)
        df['raw_pressure'] = df['imbalance'] / df['spread_sqrt'] * df['depth_ratio']
        
        # 建立时间段标签
        df['time_segment'] = label_time_segment_optimized(df, 'date')
        
        # 剔除非法时间段数据
        invalid_segments = ['OTHERS', 'OTHERS1', 'OTHERS2']
        df = df[~df['time_segment'].isin(invalid_segments)]
        
        # 使用分组应用计算因子
        result_records = []
        
        # 使用namedtuple或字典预分配内存
        for (instrument_id, trading_day, time_segment), group in df.groupby(
            ['instrument_id', 'trading_day', 'time_segment']
        ):
            factor_value = calculate_window_factor_optimized(group)
            
            # 构建结果记录
            result_records.append({
                'date': pd.to_datetime(f'{trading_day} {time_segment}'),
                'instrument_id': instrument_id,
                'factor': factor_value
            })
        
        # 批量创建DataFrame
        result_df = pd.DataFrame(result_records)
        
        # 批量查询映射数据
        instrument_ids = result_df['instrument_id'].unique().tolist()
        if instrument_ids:
            map_df = dai.query(
                'SELECT instrument, instrument_id FROM all_instruments',
                filters={'instrument_id': instrument_ids}
            ).df()
            
            # 使用merge进行连接
            if not map_df.empty:
                result_df = pd.merge(
                    result_df, 
                    map_df, 
                    how='left', 
                    on='instrument_id'
                )
        
        # 清理列
        if 'instrument' in result_df.columns:
            result_df = result_df[['date', 'instrument', 'factor']]
        else:
            # 如果没有instrument列，保持原样
            result_df = result_df[['date', 'instrument_id', 'factor']]
            result_df.rename(columns={'instrument_id': 'instrument'}, inplace=True)
        
        return result_df

    def build_factor(datasource, start_date, end_date):
        """优化构建因子"""
        sql = f"""
        SELECT
            date, instrument_id, volume, ask_price1, bid_price1,
            ask_volume1, ask_volume2, ask_volume3, ask_volume4, ask_volume5,
            bid_volume1, bid_volume2, bid_volume3, bid_volume4, bid_volume5
        FROM {datasource}
        ORDER BY instrument_id, date
        """
        
        # 使用参数化查询
        data = dai.query(
            sql, 
            filters={'date': [f'{start_date} 00:00:00', f'{end_date} 23:59:59']}
        ).df()
        
        return calc_factor(data)

    # 分月构建因子
    df_list = []
    for year in range(2023, 2025):
        for month in range(1, 13):
            start_time = datetime.now()
            
            # 计算月份的开始和结束日期
            start_date = date(year, month, 1)
            _, last_day = calendar.monthrange(year, month)
            end_date = date(year, month, last_day)
            
            try:
                factor_df = build_factor(
                    datasource,
                    start_date.strftime('%Y-%m-%d'), 
                    end_date.strftime('%Y-%m-%d')
                )
                df_list.append(factor_df)
                
                end_time = datetime.now()
                print(f'构建 {start_date} 到 {end_date}, 耗时: {end_time - start_time}')

            except Exception as e:
                print(f'处理 {year}-{month:02d} 时出错: {str(e)}')
                continue
    
    # 合并所有结果
    if df_list:
        factor_data = pd.concat(df_list, ignore_index=True)
    else:
        factor_data = pd.DataFrame(columns=['date', 'instrument', 'factor'])
    
    return factor_data

if __name__ == '__main__':
    """
    FOR Development in __main__

    This section demonstrates how to:
    1. Calculate factors using the main() function
    2. Visualize and analyze results using the factorlens module

    Tips:
        - Adjust the date range to test different time periods
        - Modify the SQL query in main() to create your own factors
        - Use factorlens to evaluate factor performance metrics
    """
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()

    datasource = 'cpt_dwc_2026_stock_hs300_snapshot'
    start_date = '2023-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"Calculating factor for period: {start_date} to {end_date}")
    data = main(datasource, start_date, end_date)
    logger.info(f"Factor data shape: {data.shape}")
    logger.info(f"\nSample data:\n{data.head()}")

    # 因子评估
    results = M.eval_dwc._latest(
        data=data
    )
